# Train the ball detector on Colab (optimization recipe)

**Setup, once per data change.** On the PC: `python finetune/ft.py colab-zip`. Upload the one file it writes,
`output/tennis_finetune_colab.zip`, to the folder `MyDrive/tennis_finetune/` in Google Drive. Open this notebook in
Colab, Runtime -> Change runtime type -> **GPU**, and Runtime -> **Run all**. Everything the notebook writes goes to
`MyDrive/tennis_finetune/runs/<TAG>/`, so a recycled runtime loses nothing: Run all again with the same `TAG` and the
run resumes from its checkpoint. `finetune/COLAB_QUICKSTART.md` has the same steps and the data downloads.

**What it trains.** `ARCH` is any network `system/tennis_tracker/ball_models.py` lists (the staging cell checks it
against the staged code). By default a new network learns from scratch on the labels, distilled from the tracker's
own detector (`DISTILL`: the teacher's soft heat maps and offsets, `finetune/training/distill.py`), with the
copy-paste ball augmentation (`copypaste` in `AUGMENT`). `TRAIN_DATA` picks the training clips: `hand` (the
hand-labelled ones), `model` (only the clips the current model labelled, `ft.py pseudo label` on the PC) or `both`
(`PSEUDO_SHARE` of every epoch from the model-labelled ones). Each choice and its evidence is in
`audit/records/opt_training.md`.

**Two kinds of run**, as before: the held-out selection run (`HOLDOUT`, scored every `SCORE_EVERY` epochs, `best.pt` the
epoch the held-out metric `recall - wrong - false_alarm` prefers), then with `SELECTION_RUN` set to that `TAG` the
all-clips run of the same recipe for the deployable weights. `LR` may be lowered (never raised) on a resume.


In [ ]:
from pathlib import Path
import shutil, sys, zipfile
from google.colab import drive
drive.mount('/content/drive')

DRIVE = Path('/content/drive/MyDrive')
DRIVE_DIR = DRIVE / 'tennis_finetune'                  # the one folder on Drive the notebook reads from and writes to
ZIP = DRIVE_DIR / 'tennis_finetune_colab.zip'
CODE_ZIP = DRIVE_DIR / 'tennis_finetune_code.zip'      # optional: ft.py colab-zip --code-only, laid over the big zip
LOCAL = Path('/content')                               # the runtime's fast local disk
DATA = LOCAL / 'tennis_finetune'                       # the unpacked workspace: videos/, labels/, pseudo/, code/, models/
WORK = DRIVE_DIR / 'runs'                              # every run's outputs, on Drive
WORK.mkdir(parents=True, exist_ok=True)
if not ZIP.is_file():
    raise SystemExit(f'Upload tennis_finetune_colab.zip to the folder MyDrive/tennis_finetune/ in your Google Drive first (looked for {ZIP})')
# Unpacked once per runtime, and again whenever the zip on Drive changes (its size and time are kept beside the copy).
stamp = DATA / '.unpacked_from'
fingerprint = f'{ZIP.stat().st_size} {ZIP.stat().st_mtime}'
if not (stamp.is_file() and stamp.read_text() == fingerprint):
    print(f'unpacking {ZIP.stat().st_size / 1e6:.0f} MB from Drive to the local disk ...')
    shutil.rmtree(DATA, ignore_errors=True)
    with zipfile.ZipFile(ZIP) as archive:
        archive.extractall(LOCAL)                      # the zip holds tennis_finetune/...
    stamp.write_text(fingerprint)
if CODE_ZIP.is_file():
    print(f'laying {CODE_ZIP.name} ({CODE_ZIP.stat().st_size / 1e3:.0f} KB) over the unpacked code and records')
    with zipfile.ZipFile(CODE_ZIP) as archive:
        archive.extractall(LOCAL)
missing = [name for name in ('clips.csv', 'review_status.json') if not (DATA / name).is_file()]
if missing:
    raise SystemExit(f'The zip is missing the workspace records {missing} (without them the trainer trusts no clip); rebuild it on the PC')
VIDEO_SUFFIXES = {'.mp4', '.mov', '.mkv', '.avi', '.m4v'}
clips = sorted(p.stem for p in (DATA / 'videos').iterdir() if p.suffix.lower() in VIDEO_SUFFIXES)
pseudo = sorted(p.name for p in (DATA / 'pseudo' / 'labels').glob('*_pseudo.csv')) if (DATA / 'pseudo' / 'labels').is_dir() else []
print(f'{len(clips)} labelled-workspace videos, {len(pseudo)} pseudo-labelled clips')
print('models:', sorted(p.name for p in (DATA / 'models').iterdir()))
print('earlier runs on Drive:', sorted(p.name for p in WORK.iterdir() if p.is_dir()))


In [ ]:
# The code comes in the zip under code/, laid out as the repository is; ft.py colab-zip derives the list from the
# trainer's imports and writes it to code/code_manifest.json, which is checked here.
import json
CODE = DATA / 'code'
if not (CODE / 'code_manifest.json').is_file():
    raise SystemExit('The zip has no code/code_manifest.json; rebuild it on the PC with the current repo')
REQUIRED = json.loads((CODE / 'code_manifest.json').read_text(encoding='utf-8'))
missing = [name for name in REQUIRED if not (CODE / name).is_file()]
if missing:
    raise SystemExit(f'The zip is missing code files {missing}; rebuild it on the PC with the current repo')
!pip -q install opencv-python-headless numpy
import torch; print('torch', torch.__version__, 'cuda', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '-')


In [ ]:
#@title Training settings
# The network (system/tennis_tracker/ball_models.py; checked below against the staged code).
ARCH = "heatmap_s4"         #@param ["heatmap_s4", "heatmap_lite", "stream_s4", "wasb_5f", "gridtracknet"]
MODE = "scratch"            #@param ["finetune", "scratch"]
# The fine-tune start (the architecture's own file): a name in the zip's models/ folder or a path under runs/ (say scratch1/best.pt).
START_WEIGHTS = "ball_detector.pt"  #@param {type:"string"}
# Distillation from TEACHER (a name in models/ or a path under runs/): its soft heat maps and offsets beside the labels.
DISTILL = True              #@param {type:"boolean"}
TEACHER = "ball_detector.pt"  #@param {type:"string"}
DISTILL_LOSS = "qfl"        #@param ["qfl", "mse"]
DISTILL_WEIGHT = 1.0        #@param {type:"number"}
# online: the teacher answers every augmented batch; cache: its answers are precomputed once on the local disk.
TEACHER_MAPS = "online"     #@param ["online", "cache"]
# Masked Generative Distillation of the decoder features (0 = off; the paper's detection setting is 2e-5; online only).
MGD_WEIGHT = 0.0            #@param {type:"number"}
# What it trains on: hand = your hand-labelled clips; model = only the clips your current model labelled (ft.py pseudo
# label on the PC); both = the two mixed. Whichever it is, the held-out score is always against hand labels.
TRAIN_DATA = "hand"         #@param ["hand", "model", "both"]
# both only: the model-labelled clips' share of every epoch's draws.
PSEUDO_SHARE = 0.3          #@param {type:"number"}
# Space-separated, any of: basic strong translate reverse compress copypaste.
AUGMENT = "basic translate copypaste"  #@param {type:"string"}
# A held-out set from holdouts.json (whole recordings); empty = use VAL_CLIPS.
HOLDOUT = "mixed_v1"        #@param {type:"string"}
# Space-separated clip stems to hold out when HOLDOUT is empty; empty = video10.
VAL_CLIPS = ""              #@param {type:"string"}
# The TAG of a finished held-out run on Drive: set, this is the all-clips run of that run's recipe (nothing held out or scored).
SELECTION_RUN = ""          #@param {type:"string"}
# A ceiling, not a target: best.pt is kept by the held-out metric. With SELECTION_RUN, 0 = that run's best epoch.
EPOCHS = 80                 #@param {type:"integer"}
BATCH_SIZE = 4              #@param {type:"integer"}
# Adadelta; 0.3 for a gentle fine-tune. On a resume it may be lowered (not raised): the run goes on at the new rate from its next epoch.
LR = 1.0                    #@param {type:"number"}
SEED = 7                    #@param {type:"integer"}
# Score the held-out set every K epochs (a few minutes each).
SCORE_EVERY = 2             #@param {type:"integer"}
OVERSAMPLE_ABSENT = 1.0     #@param {type:"number"}
# Data-loading processes; 0 = one per CPU the runtime has.
WORKERS = 0                 #@param {type:"integer"}
# The run's folder under MyDrive/tennis_finetune/runs/; one that holds a checkpoint.pt is resumed.
TAG = "opt1"                #@param {type:"string"}

import json, os, subprocess
RUN = WORK / TAG
# The architectures the staged code knows, asked of it (not of this list), so a new one needs no notebook edit.
known = subprocess.run([sys.executable, '-c', 'import finetune; from tennis_tracker.ball_models import ARCHITECTURES, '
                        'weights_suffix; print(" ".join(f"{a}={weights_suffix(a)}" for a in ARCHITECTURES))'],
                       cwd=str(CODE), capture_output=True, text=True, check=True).stdout.split()
SUFFIXES = dict(item.split('=') for item in known)
if ARCH not in SUFFIXES:
    raise SystemExit(f'ARCH {ARCH!r} is not in the staged ball_models (it knows {sorted(SUFFIXES)}); rebuild the zip with a repo that has it')
SUFFIX = SUFFIXES[ARCH]                                # the weights suffix every file the run writes carries

def weights_file(name, what):
    # A name in the zip's models/ folder, or a path under runs/ on Drive.
    for path in (DATA / 'models' / name, WORK / name):
        if path.is_file():
            return path
    raise SystemExit(f'{what} {name!r} is neither in the zip ({DATA / "models"}) nor under {WORK}')

START = weights_file(START_WEIGHTS, 'START_WEIGHTS')
if MODE == 'finetune':
    start_arch = 'gridtracknet' if START.suffix == '.npz' else torch.load(START, map_location='cpu', weights_only=True).get('arch')
    if start_arch != ARCH:
        raise SystemExit(f'a {ARCH} fine-tune starts from a {ARCH} file, not {START.name} ({start_arch}); set MODE to scratch')
all_clips = bool(SELECTION_RUN.strip())
if all_clips:
    SELECTION = WORK / SELECTION_RUN.strip()
    if SELECTION_RUN.strip() == TAG:
        raise SystemExit(f'SELECTION_RUN and TAG are both {TAG!r}: the all-clips run needs a folder of its own')
    if not (SELECTION / 'winner.json').is_file():
        raise SystemExit(f'SELECTION_RUN {SELECTION_RUN!r} is not a finished run on Drive: no {SELECTION / "winner.json"}')
    selected = json.loads((SELECTION / 'config.json').read_text(encoding='utf-8')).get('arch') or 'gridtracknet'
    if selected != ARCH:
        raise SystemExit(f'SELECTION_RUN {SELECTION_RUN!r} trained {selected}: set ARCH to it')
RUN.mkdir(parents=True, exist_ok=True)
workers = WORKERS or max(1, os.cpu_count() or 1)
print(f'{os.cpu_count()} CPUs, {workers} loader workers, GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none')
CACHE = LOCAL / 'cache' / 'data'                       # the frame cache (and teacher maps), on the fast local disk
cmd = [sys.executable, '-u', '-m', 'finetune.training.train_gridtracknet',
       '--videos', str(DATA / 'videos'), '--labels', str(DATA / 'labels'),
       '--weights', str(START), '--arch', ARCH, '--save', str(RUN / ('winner' + SUFFIX)),
       '--data-dir', str(CACHE), '--run-dir', str(RUN),
       '--batch-size', str(BATCH_SIZE), '--lr', str(LR),
       '--workers', str(workers), '--prep-workers', str(workers),
       '--seed', str(SEED), '--augment', *AUGMENT.split(),
       '--oversample-absent', str(OVERSAMPLE_ABSENT)]
if DISTILL:
    cmd += ['--distill', '--teacher', str(weights_file(TEACHER, 'TEACHER')), '--distill-loss', DISTILL_LOSS,
            '--distill-weight', str(DISTILL_WEIGHT), '--teacher-maps', TEACHER_MAPS, '--mgd-weight', str(MGD_WEIGHT)]
    print(f'distilling from {TEACHER}: {DISTILL_LOSS} x{DISTILL_WEIGHT}, teacher maps {TEACHER_MAPS}'
          + (f', MGD x{MGD_WEIGHT}' if MGD_WEIGHT else ''))
if TRAIN_DATA not in ('hand', 'model', 'both'):
    raise SystemExit(f'TRAIN_DATA is hand, model or both, not {TRAIN_DATA!r}')
if TRAIN_DATA == 'hand':
    print('training data: the hand-labelled clips')
elif not pseudo:
    raise SystemExit(f'TRAIN_DATA {TRAIN_DATA!r} needs model-labelled clips, and the zip holds none: on the PC run '
                     'ft.py pseudo add and ft.py pseudo label, then ft.py colab-zip, and upload the new zip')
else:
    share = 1.0 if TRAIN_DATA == 'model' else PSEUDO_SHARE
    cmd += ['--pseudo', str(DATA / 'pseudo'), '--pseudo-share', str(share)]
    print(f'training data: {len(pseudo)} model-labelled clips only' if TRAIN_DATA == 'model' else
          f'training data: the hand-labelled clips and {len(pseudo)} model-labelled clips as {share:.0%} of every epoch')
if not (all_clips and EPOCHS == 0):          # 0 with SELECTION_RUN: the trainer takes the selection run's best epoch
    cmd += ['--epochs', str(EPOCHS)]
if all_clips:
    cmd += ['--all-clips', '--selection-run', str(SELECTION)]
    print(f'all-clips run: every clip, nothing held out, following the recipe of {SELECTION_RUN.strip()}')
else:
    cmd += ['--score-every', str(SCORE_EVERY)]
    if HOLDOUT.strip():
        cmd += ['--holdout', HOLDOUT.strip()]
    elif VAL_CLIPS.strip():
        cmd += ['--val-clips', *VAL_CLIPS.split()]
    print(f'held-out selection run: {HOLDOUT.strip() or VAL_CLIPS.split() or ["video10"]}, scored every {SCORE_EVERY} epochs')
if MODE == 'scratch':
    cmd.append('--from-scratch')
if (RUN / 'checkpoint.pt').is_file():
    cmd.append('--resume')
    print(f'resuming {TAG} from its checkpoint on Drive (delete {RUN} or change TAG for a fresh run)')
    if (RUN / 'config.json').is_file():
        config = json.loads((RUN / 'config.json').read_text(encoding='utf-8'))
        rate = (config.get('lr_schedule') or [{'lr': config.get('lr')}])[-1]['lr']    # the rate in force
        if LR != rate:
            cmd.append('--lr-drop')
            print(f'the learning rate drops from {rate} to {LR} for the epochs from here on (a raise is refused by the trainer)')
print(' '.join(cmd))


In [ ]:
# Runs the trainer and streams its log here and to Drive (RUN/colab.log). The trainer prints a step line every few
# seconds (loss, it/s, eta) and, after every epoch, one summary line: train loss, held-out recall / wrong / false alarm
# and which epoch best.npz holds. On a scored epoch the held-out table so far is reprinted underneath, with the
# epoch best.npz holds marked. An all-clips run (SELECTION_RUN set) holds nothing out, so its epoch lines carry the
# train loss alone and no table follows. Interrupting the cell stops the trainer too; the run resumes from its checkpoint.
import csv, re, subprocess

def held_out_table(best_epoch):
    """The scored rows of results.csv so far (an unscored epoch leaves the metric columns empty), or
    what went wrong reading them: a table that cannot be read must never stop the log."""
    try:
        with (RUN / 'results.csv').open(newline='') as handle:
            scored = [row for row in csv.DictReader(handle) if row.get('recall')]
        lines = ['    epoch  train_loss  recall  wrong  falseAlarm  med_px  min']
        for row in scored:
            mark = f' <- best{SUFFIX}' if int(row['epoch']) == best_epoch else ''
            lines.append(f"    {int(row['epoch']):5d}  {float(row['train_loss']):.6f}  {float(row['recall']):.3f}"
                         f"  {float(row['wrong']):.3f}  {float(row['false_alarm']):.3f}      {float(row['median_px']):.1f}"
                         f"  {float(row['seconds']) / 60:3.0f}{mark}")
        return '\n'.join(lines)
    except (OSError, ValueError, KeyError) as error:
        return f'    (results.csv not readable yet: {error})'

process = subprocess.Popen(cmd, cwd=str(CODE), stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
try:
    with (RUN / 'colab.log').open('a', encoding='utf-8') as log:
        for line in process.stdout:
            print(line, end='')
            log.write(line)
            log.flush()
            best = re.search(r'best epoch (\d+)', line)
            if line.startswith('[train] epoch') and best:
                print(held_out_table(int(best.group(1))), flush=True)
finally:
    if process.poll() is None:
        process.terminate()
    process.wait()
print('exit code', process.returncode)

In [ ]:
# The whole run at a glance: the per-epoch table, the curves and the winner decision. An all-clips run has only the
# train loss to show: nothing was held out, so results.csv has no validation loss and no scored rows.
import json
import pandas as pd
import matplotlib.pyplot as plt
results = pd.read_csv(RUN / 'results.csv')
print(results.to_string(index=False))
scored = results.dropna(subset=['recall'])
fig, axes = plt.subplots(1, 3 if len(scored) else 1, figsize=(15 if len(scored) else 5, 3.5), squeeze=False)
axes = axes[0]
axes[0].plot(results['epoch'], results['train_loss'], label='train')
if results['val_loss'].notna().any():
    axes[0].plot(results['epoch'], results['val_loss'], label='val')
axes[0].set_title('loss'); axes[0].legend()
if len(scored):
    axes[1].plot(scored['epoch'], scored['recall'], marker='o'); axes[1].set_title('held-out recall')
    axes[2].plot(scored['epoch'], scored['false_alarm'], marker='o', label='false alarm')
    axes[2].plot(scored['epoch'], scored['wrong'], marker='o', label='wrong')
    axes[2].set_title('held-out false alarm / wrong'); axes[2].legend()
for ax in axes:
    ax.set_xlabel('epoch'); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()
if (RUN / 'winner.json').is_file():
    print(json.dumps(json.loads((RUN / 'winner.json').read_text()), indent=2))
else:
    print('no winner.json yet: the run has not reached its last epoch (resume it with the same TAG)')
if SELECTION_RUN.strip():
    print(f'\nWeights on Drive: {RUN}/winner{SUFFIX} (last{SUFFIX}, the final epoch, trained on every clip following {SELECTION_RUN.strip()}).')
    print('They saw every labelled clip, so no held-out number can be put on them: the test left is the tracker on unlabelled footage.')
else:
    print(f'\nWeights on Drive: {RUN}/best{SUFFIX} (the epoch the detector metric chose), last{SUFFIX} (the final epoch), winner{SUFFIX} (the saved result).')
    print('Back on the PC, score any of them with  evaluate_archive.py --archive finetune --mode raw --holdout mixed_v1 --gridtracknet-weights <file>')

## After the run

1. `best.pt` (the epoch the held-out metric chose), `last.pt` and `winner.pt` are in `MyDrive/tennis_finetune/runs/<TAG>/`.
   Score one on the PC: `evaluate_archive.py --archive finetune --mode raw --holdout mixed_v1 --gridtracknet-weights <best.pt>`.
2. The deployable weights: set `SELECTION_RUN` to this `TAG`, a new `TAG`, `EPOCHS` 0, keep `ARCH` and the other settings,
   and Run all: the all-clips run follows the recipe on every clip.
3. `ft.py promote --weights <winner.pt>` makes it the tracker's model (with a backup); that is your call.
